# 01 - Data Collection

This notebook collects Content Security Policy headers from Tranco top websites using **requests only**. It handles redirects, timeouts, exceptions, one retry, and a polite delay between requests.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.csp_collection import collect_csp_dataset, load_tranco_domains

DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

## Load Tranco Domains

For final submission, download a Tranco CSV list from https://tranco-list.eu/ and place it in the `data` folder. Use 1000 to 3000 domains. A small sample file is included for testing.

In [ ]:
TRANCO_FILE = DATA_DIR / 'tranco.csv'  # your real Tranco CSV file
DOMAIN_LIMIT = 1000

domains = load_tranco_domains(TRANCO_FILE, limit=DOMAIN_LIMIT)
len(domains), domains[:5]

## Collect CSP Headers

The delay is set between 0.5 and 1 second to reduce blocking. Timeout is 12 seconds, which satisfies the 10 to 15 second requirement.

In [ ]:
df = collect_csp_dataset(
    domains,
    delay_seconds=0.7,
    timeout=12,
    retries=1,
    skip_unreachable=True,
)

df.head()

In [ ]:
df[['domain', 'csp_header', 'report_only_header', 'status_code', 'has_csp']].to_csv(DATA_DIR / 'csp_dataset.csv', index=False)
df[['domain', 'csp_header', 'report_only_header', 'status_code', 'has_csp']].head()

In [ ]:
df['has_csp'].value_counts(dropna=False)